[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/37.%20The%20Equipment%20Selection%20Problem%20%28RTG%20vs.%20RMG%20vs.%20Straddle%29/P37-Tier-3_executed.ipynb)

---

# 37. The Equipment Selection Problem (RTG vs. RMG vs. Straddle)
## Tier 3: Artificial Bee Colony Optimization

### Goal and workflow

This notebook implements the Artificial Bee Colony (ABC) metaheuristic for selecting quantities of three types of container-handling equipment: rubber-tired gantry cranes (RTGs), rail-mounted gantry cranes (RMGs), and straddle carriers. The algorithm searches for a high-scoring equipment mix while accounting for capital cost, productivity, flexibility, energy use, and penalty terms for constraint violations.

The workflow follows the ABC sequence: define equipment data, initialize food sources, explore them in the employed-bee phase, select and explore sources in the onlooker-bee phase, replace stagnant sources in the scout-bee phase, and track the best fitness and population diversity. The notebook then evaluates the resulting solution and examines how colony size affects a set of runs.

### Background: what the bees represent

In the algorithm's analogy, a **food source** is a candidate solution, and its **nectar amount** is represented by its fitness. Employed bees explore around current sources. Onlooker bees choose sources with probabilities related to their fitness and explore them. A source that has not improved for a specified number of trials can be abandoned; a scout bee then introduces a randomly generated source. These are search operations, not claims about the behavior of real bees.

A candidate is encoded as a three-component vector

$$\mathbf{x} = (x_{\mathrm{RTG}}, x_{\mathrm{RMG}}, x_{\mathrm{Straddle}}).$$

The search moves through real-valued vectors, while evaluation converts each component to a nonnegative integer equipment count using integer truncation. Thus, the decision being evaluated is a discrete equipment mix even though ABC's neighborhood perturbations are continuous.

### Fitness model

The algorithm maximizes a weighted score. For integer counts $n_i$, let $C$ be total capital cost, $P$ total productivity, $F$ the equipment-count-weighted average flexibility, and $E$ the average energy consumption per container. The objective score before penalties is

$$S = w_C\frac{1}{1+C} + w_P\frac{P}{1000} + w_F F + w_E\frac{1}{1+E}.$$

The fitness used by the search is $S$ minus penalties for exceeding the budget, falling short of the productivity requirement, or selecting fewer than five units. In this example, the default weights are $w_C=0.3$, $w_P=0.4$, $w_F=0.2$, and $w_E=0.1$. The cost term rewards lower capital cost, while the productivity, flexibility, and sustainability terms reward higher values. Sustainability is represented by the inverse of average energy use, so lower energy use gives a higher score.

### ABC search mechanics

For a source $i$, a standard ABC-style neighbor changes one coordinate at a time. The implementation selects coordinate $j$ and adds a normally distributed perturbation, then clips that coordinate to its bounds. A greedy acceptance rule keeps the candidate if its fitness is higher. Onlooker selection uses roulette-wheel probabilities $p_i = q_i / \sum_k q_k$, where $q_i$ are nonnegative values derived from source fitness. If a source's trial count exceeds the abandonment limit, the scout phase replaces it with a random source.

The initial population contains half the configured colony as employed bees and half as onlooker bees. Each iteration performs the employed, onlooker, and scout phases in that order. The stored best fitness is the best observed during the run, and the convergence check looks for a small range in the recent best-fitness history.

If the colony size is $N$ and the number of iterations is $T$, each iteration evaluates a number of candidates proportional to $N$. With three equipment types, the search work is therefore approximately $O(TN)$ fitness evaluations, apart from the constant cost of evaluating the three equipment types. ABC is stochastic and does not certify that its best observed solution is globally optimal.

### Model assumptions and scope

- Equipment quantities are integer counts, and each count is bounded by the stated search bounds.
- Capital cost, productivity, flexibility, and energy use are additive or count-weighted as implemented below. Capital cost is the only cost included in the budget calculation; operating and maintenance costs are stored in the equipment data but are not included in this fitness model.
- The productivity requirement, budget limit, and minimum fleet size are handled by penalty terms rather than by a separate feasibility solver. A high fitness is therefore not, by itself, proof that every constraint is satisfied; the reported solution is checked explicitly.
- Flexibility and energy are aggregated as weighted averages across equipment units. The model does not represent interactions between equipment types, layout, staffing, reliability, service demand over time, financing, or uncertainty in the input data.
- Objective weights and penalty magnitudes encode modeling preferences and units. The resulting weighted score is useful for this specified example, but it is not a unit-free measure of real-world value.

### Example configuration and interpretation

The main run uses a colony size of 40, up to 500 iterations, an abandonment limit of 50, a convergence window of 20, a capital budget of $50 million, and a minimum productivity of 500 containers per hour. The quantity bounds are 0–25 RTGs, 0–18 RMGs, and 0–40 straddle carriers. The random-number generators are seeded to make a run reproducible in a fixed software environment.

The fitness history shows the best fitness observed by each iteration; the diversity history summarizes the spread of the bee positions. Neither curve proves global optimality. The sensitivity experiment varies colony size and reports the observed outcomes for those runs. Because ABC is stochastic, a single run per setting is an illustrative comparison, not a statistical performance guarantee.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

np.random.seed(42)
random.seed(42)

print("Equipment Selection - Artificial Bee Colony Optimization")
print("Population-based stochastic search")

### Equipment data

Each equipment type has attributes used either in the fitness calculation or in the descriptive output. Costs are measured in millions of US dollars; operating and maintenance costs are annual values. Productivity is measured in containers per hour, flexibility is a score from 0 to 1, footprint is in square meters, and energy consumption is in kWh per container. In this model, capital cost, productivity, flexibility, and energy consumption contribute to fitness; the other attributes are retained as descriptive data.

In [ ]:
@dataclass
class EquipmentType:
    name: str
    capital_cost: float  # Million USD
    operating_cost: float  # Million USD per year
    productivity: float  # Containers per hour
    flexibility: float  # Scale from 0 to 1
    maintenance: float  # Million USD per year
    footprint: float  # Square meters
    energy_consumption: float  # kWh per container


equipment_types = {
    "RTG": EquipmentType(
        name="Rubber-Tired Gantry",
        capital_cost=2.5,
        operating_cost=0.8,
        productivity=25,
        flexibility=0.8,
        maintenance=0.3,
        footprint=200,
        energy_consumption=3.2,
    ),
    "RMG": EquipmentType(
        name="Rail-Mounted Gantry",
        capital_cost=3.2,
        operating_cost=0.6,
        productivity=35,
        flexibility=0.4,
        maintenance=0.25,
        footprint=150,
        energy_consumption=2.8,
    ),
    "Straddle": EquipmentType(
        name="Straddle Carrier",
        capital_cost=4.1,
        operating_cost=1.1,
        productivity=20,
        flexibility=0.9,
        maintenance=0.4,
        footprint=100,
        energy_consumption=4.5,
    ),
}

print("Equipment types defined:")
for equipment_key, equipment in equipment_types.items():
    print(f"{equipment_key}: {equipment.name}")
    print(f"  Capital cost: ${equipment.capital_cost}M")
    print(f"  Operating cost: ${equipment.operating_cost}M/year")
    print(f"  Productivity: {equipment.productivity} containers/hour")
    print(f"  Flexibility: {equipment.flexibility:.1f}")
    print()

### Artificial Bee Colony algorithm implementation

The code below keeps the algorithm's phases visible so their roles can be followed independently. `evaluate_fitness` translates a candidate vector into integer counts and scores it. The three bee-phase methods update candidate sources, while `optimize` controls their order and records the run history. The source positions remain real-valued during search; fitness evaluation applies integer truncation to obtain the equipment counts being scored.

In [ ]:
@dataclass
class ABCConfig:
    colony_size: int = 40
    max_iterations: int = 500
    abandonment_limit: int = 50
    convergence_window: int = 20


@dataclass
class BeeSolution:
    position: np.ndarray  # [RTG, RMG, Straddle]
    fitness: float
    trial_count: int = 0


class ArtificialBeeColonyOptimizer:
    def __init__(
        self,
        equipment_types: Dict[str, EquipmentType],
        config: ABCConfig,
        budget_limit: float = 50.0,
        min_productivity: float = 500,
        objective_weights: Optional[Dict[str, float]] = None,
    ):
        if config.colony_size < 2 or config.colony_size % 2 != 0:
            raise ValueError("colony_size must be a positive even integer of at least 2")
        if config.max_iterations < 1:
            raise ValueError("max_iterations must be at least 1")
        if config.convergence_window < 1:
            raise ValueError("convergence_window must be at least 1")

        self.equipment_types = equipment_types
        self.config = config
        self.budget_limit = budget_limit
        self.min_productivity = min_productivity

        self.objective_weights = objective_weights or {
            "cost": 0.3,
            "productivity": 0.4,
            "flexibility": 0.2,
            "sustainability": 0.1,
        }
        required_weights = {"cost", "productivity", "flexibility", "sustainability"}
        if set(self.objective_weights) != required_weights:
            raise ValueError(f"objective_weights must contain exactly {required_weights}")

        self.bounds = {
            "RTG": (0, 25),
            "RMG": (0, 18),
            "Straddle": (0, 40),
        }
        self.employed_bees: List[BeeSolution] = []
        self.onlooker_bees: List[BeeSolution] = []
        self.best_solution: Optional[np.ndarray] = None
        self.best_fitness = -np.inf
        self.fitness_history: List[float] = []
        self.diversity_history: List[float] = []
        self.convergence_iteration: Optional[int] = None

    def evaluate_fitness(self, position: np.ndarray) -> float:
        """Score the integer equipment mix represented by a search position."""
        position = np.asarray(position, dtype=float)
        if position.shape != (3,):
            raise ValueError("position must contain exactly three equipment quantities")

        bounds_list = [self.bounds["RTG"], self.bounds["RMG"], self.bounds["Straddle"]]
        counts = [
            int(np.clip(value, lower, upper))
            for value, (lower, upper) in zip(position, bounds_list)
        ]
        rtg, rmg, straddle = counts

        total_cost = (
            self.equipment_types["RTG"].capital_cost * rtg
            + self.equipment_types["RMG"].capital_cost * rmg
            + self.equipment_types["Straddle"].capital_cost * straddle
        )
        total_productivity = (
            self.equipment_types["RTG"].productivity * rtg
            + self.equipment_types["RMG"].productivity * rmg
            + self.equipment_types["Straddle"].productivity * straddle
        )
        total_units = rtg + rmg + straddle

        if total_units == 0:
            return -1e6

        weighted_flexibility = (
            self.equipment_types["RTG"].flexibility * rtg
            + self.equipment_types["RMG"].flexibility * rmg
            + self.equipment_types["Straddle"].flexibility * straddle
        ) / total_units
        total_energy = (
            self.equipment_types["RTG"].energy_consumption * rtg
            + self.equipment_types["RMG"].energy_consumption * rmg
            + self.equipment_types["Straddle"].energy_consumption * straddle
        )
        average_energy = total_energy / total_units
        sustainability_score = 1.0 / (1.0 + average_energy)

        penalty = 0.0
        if total_cost > self.budget_limit:
            penalty += (total_cost - self.budget_limit) * 10
        if total_productivity < self.min_productivity:
            penalty += (self.min_productivity - total_productivity) * 5
        if total_units < 5:
            penalty += (5 - total_units) * 2

        score = (
            self.objective_weights["cost"] / (1.0 + total_cost)
            + self.objective_weights["productivity"] * (total_productivity / 1000.0)
            + self.objective_weights["flexibility"] * weighted_flexibility
            + self.objective_weights["sustainability"] * sustainability_score
        )
        return score - penalty

    def initialize_population(self) -> None:
        """Create random employed sources and matching onlooker positions."""
        self.employed_bees = []
        self.onlooker_bees = []
        self.best_solution = None
        self.best_fitness = -np.inf
        self.fitness_history = []
        self.diversity_history = []
        self.convergence_iteration = None

        for _ in range(self.config.colony_size // 2):
            position = np.array(
                [
                    np.random.uniform(*self.bounds["RTG"]),
                    np.random.uniform(*self.bounds["RMG"]),
                    np.random.uniform(*self.bounds["Straddle"]),
                ]
            )
            fitness = self.evaluate_fitness(position)
            self.employed_bees.append(BeeSolution(position, fitness))
            if fitness > self.best_fitness:
                self.best_fitness = fitness
                self.best_solution = position.copy()

        for employed_bee in self.employed_bees:
            self.onlooker_bees.append(
                BeeSolution(employed_bee.position.copy(), employed_bee.fitness)
            )

    def employed_bee_phase(self) -> None:
        """Explore one-coordinate neighborhoods of employed sources."""
        bounds_list = [self.bounds["RTG"], self.bounds["RMG"], self.bounds["Straddle"]]
        for bee in self.employed_bees:
            new_position = bee.position.copy()
            coordinate = np.random.randint(0, 3)
            lower, upper = bounds_list[coordinate]
            scale = (upper - lower) * 0.1
            new_position[coordinate] += np.random.normal(0, scale)
            new_position[coordinate] = np.clip(new_position[coordinate], lower, upper)
            new_fitness = self.evaluate_fitness(new_position)

            if new_fitness > bee.fitness:
                bee.position = new_position
                bee.fitness = new_fitness
                bee.trial_count = 0
                if new_fitness > self.best_fitness:
                    self.best_fitness = new_fitness
                    self.best_solution = new_position.copy()
            else:
                bee.trial_count += 1

    def onlooker_bee_phase(self) -> None:
        """Select employed sources by roulette wheel and explore them."""
        fitness_values = np.array([bee.fitness for bee in self.employed_bees])
        shifted_fitness = fitness_values - min(0.0, float(np.min(fitness_values))) + 1e-6
        probabilities = shifted_fitness / np.sum(shifted_fitness)
        bounds_list = [self.bounds["RTG"], self.bounds["RMG"], self.bounds["Straddle"]]

        for onlooker_bee in self.onlooker_bees:
            selected_index = np.random.choice(len(self.employed_bees), p=probabilities)
            selected_bee = self.employed_bees[selected_index]
            new_position = selected_bee.position.copy()
            coordinate = np.random.randint(0, 3)
            lower, upper = bounds_list[coordinate]
            scale = (upper - lower) * 0.1
            new_position[coordinate] += np.random.normal(0, scale)
            new_position[coordinate] = np.clip(new_position[coordinate], lower, upper)
            new_fitness = self.evaluate_fitness(new_position)

            if new_fitness > selected_bee.fitness:
                selected_bee.position = new_position
                selected_bee.fitness = new_fitness
                selected_bee.trial_count = 0
                if new_fitness > self.best_fitness:
                    self.best_fitness = new_fitness
                    self.best_solution = new_position.copy()
            else:
                selected_bee.trial_count += 1

            onlooker_bee.position = selected_bee.position.copy()
            onlooker_bee.fitness = selected_bee.fitness

    def scout_bee_phase(self) -> None:
        """Replace employed sources that have exceeded the trial limit."""
        for bee in self.employed_bees:
            if bee.trial_count > self.config.abandonment_limit:
                bee.position = np.array(
                    [
                        np.random.uniform(*self.bounds["RTG"]),
                        np.random.uniform(*self.bounds["RMG"]),
                        np.random.uniform(*self.bounds["Straddle"]),
                    ]
                )
                bee.fitness = self.evaluate_fitness(bee.position)
                bee.trial_count = 0
                if bee.fitness > self.best_fitness:
                    self.best_fitness = bee.fitness
                    self.best_solution = bee.position.copy()

    def calculate_diversity(self) -> float:
        """Measure mean Euclidean distance from the population mean position."""
        all_bees = self.employed_bees + self.onlooker_bees
        if len(all_bees) < 2:
            return 0.0
        positions = np.array([bee.position for bee in all_bees])
        mean_position = np.mean(positions, axis=0)
        return float(np.mean(np.linalg.norm(positions - mean_position, axis=1)))

    def check_convergence(self) -> bool:
        """Check whether recent best-fitness values vary by less than tolerance."""
        window = self.config.convergence_window
        if len(self.fitness_history) < window:
            return False
        recent_fitness = self.fitness_history[-window:]
        return max(recent_fitness) - min(recent_fitness) < 1e-6

    def optimize(self) -> Tuple[np.ndarray, List[float], List[float]]:
        """Run the employed, onlooker, and scout phases in sequence."""
        print(f"Starting ABC optimization with {self.config.colony_size} bees...")
        self.initialize_population()
        print(f"Employed bees: {len(self.employed_bees)}")
        print(f"Onlooker bees: {len(self.onlooker_bees)}")
        print(f"Maximum iterations: {self.config.max_iterations}")
        print(f"Initial best fitness: {self.best_fitness:.4f}")

        start_time = time.perf_counter()
        for iteration in range(self.config.max_iterations):
            self.employed_bee_phase()
            self.onlooker_bee_phase()
            self.scout_bee_phase()

            self.fitness_history.append(self.best_fitness)
            self.diversity_history.append(self.calculate_diversity())

            if (iteration + 1) % 50 == 0:
                print(
                    f"Iteration {iteration + 1}: best fitness = {self.best_fitness:.4f}, "
                    f"diversity = {self.diversity_history[-1]:.3f}"
                )

            if iteration > 100 and self.check_convergence():
                self.convergence_iteration = iteration + 1
                print(f"Convergence criterion met at iteration {iteration + 1}")
                break

        elapsed_time = time.perf_counter() - start_time
        print(f"Optimization completed in {elapsed_time:.2f} seconds")
        print(f"Best fitness observed: {self.best_fitness:.4f}")
        return self.best_solution.copy(), self.fitness_history, self.diversity_history


print("Artificial Bee Colony implementation is ready.")

### Run the ABC optimization

The main run instantiates the stated configuration and calls `optimize`. The returned vector is converted to the same integer counts used by `evaluate_fitness`; those counts are then used to report the model's physical quantities and constraint status. The optimizer searches for a high-fitness solution, so the output is an ABC result, not a mathematical certificate of global optimality.

In [ ]:
config = ABCConfig(
    colony_size=40,
    max_iterations=500,
    abandonment_limit=50,
    convergence_window=20,
)

optimizer = ArtificialBeeColonyOptimizer(
    equipment_types=equipment_types,
    config=config,
    budget_limit=50.0,
    min_productivity=500,
)

best_solution, fitness_history, diversity_history = optimizer.optimize()

best_rtg, best_rmg, best_straddle = [
    int(np.clip(value, lower, upper))
    for value, (lower, upper) in zip(
        best_solution,
        [optimizer.bounds["RTG"], optimizer.bounds["RMG"], optimizer.bounds["Straddle"]],
    )
]

total_units = best_rtg + best_rmg + best_straddle
total_cost = (
    best_rtg * equipment_types["RTG"].capital_cost
    + best_rmg * equipment_types["RMG"].capital_cost
    + best_straddle * equipment_types["Straddle"].capital_cost
)
total_productivity = (
    best_rtg * equipment_types["RTG"].productivity
    + best_rmg * equipment_types["RMG"].productivity
    + best_straddle * equipment_types["Straddle"].productivity
)
weighted_flexibility = (
    best_rtg * equipment_types["RTG"].flexibility
    + best_rmg * equipment_types["RMG"].flexibility
    + best_straddle * equipment_types["Straddle"].flexibility
) / total_units
total_energy = (
    best_rtg * equipment_types["RTG"].energy_consumption
    + best_rmg * equipment_types["RMG"].energy_consumption
    + best_straddle * equipment_types["Straddle"].energy_consumption
)
average_energy = total_energy / total_units
reported_fitness = optimizer.evaluate_fitness(best_solution)

assert total_units >= 1
assert np.isclose(reported_fitness, optimizer.best_fitness, rtol=1e-12, atol=1e-12)

print("\n=== BEST OBSERVED EQUIPMENT MIX ===")
print(f"RTG units: {best_rtg}")
print(f"RMG units: {best_rmg}")
print(f"Straddle units: {best_straddle}")
print(f"Total equipment: {total_units}")
print(f"Total capital cost: ${total_cost:.1f}M")
print(f"Total productivity: {total_productivity:.0f} containers/hour")
print(f"Average flexibility: {weighted_flexibility:.2f}")
print(f"Average energy consumption: {average_energy:.2f} kWh/container")
print(f"Fitness: {reported_fitness:.4f}")
print(f"Budget constraint satisfied: {total_cost <= optimizer.budget_limit}")
print(f"Productivity constraint satisfied: {total_productivity >= optimizer.min_productivity}")
print(f"Minimum fleet-size condition satisfied: {total_units >= 5}")

### Performance analysis and visualization

The first two panels show the best fitness and population diversity over iterations. The fitness curve is based on the best solution observed so far, while diversity measures the mean distance of bee positions from their population mean. The third panel displays the selected equipment counts. The final panel gives an at-a-glance profile of four outcomes; its values are scaled and capped for display and are not the weighted terms used directly in the fitness function.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Artificial Bee Colony Optimization Analysis", fontsize=16, fontweight="bold")

iterations = np.arange(1, len(fitness_history) + 1)
ax_fitness = axes[0, 0]
ax_fitness.plot(iterations, fitness_history, color="steelblue", linewidth=2, label="Best fitness")
ax_fitness.set_xlabel("Iteration")
ax_fitness.set_ylabel("Fitness value")
ax_fitness.set_title("Best Fitness by Iteration")
ax_fitness.grid(True, alpha=0.3)
ax_fitness.legend()
if optimizer.convergence_iteration is not None:
    ax_fitness.axvline(
        optimizer.convergence_iteration,
        color="darkorange",
        linestyle="--",
        label=f"Convergence criterion: {optimizer.convergence_iteration}",
    )
    ax_fitness.legend()

ax_diversity = axes[0, 1]
ax_diversity.plot(
    iterations,
    diversity_history,
    color="seagreen",
    linewidth=2,
    label="Population diversity",
)
ax_diversity.set_xlabel("Iteration")
ax_diversity.set_ylabel("Mean distance from population mean")
ax_diversity.set_title("Population Diversity by Iteration")
ax_diversity.grid(True, alpha=0.3)
ax_diversity.legend()

equipment_names = ["RTG", "RMG", "Straddle"]
equipment_counts = [best_rtg, best_rmg, best_straddle]
bars = axes[1, 0].bar(
    equipment_names,
    equipment_counts,
    color=["steelblue", "darkorange", "seagreen"],
    alpha=0.8,
)
axes[1, 0].set_ylabel("Number of units")
axes[1, 0].set_title("Best Observed Equipment Mix")
axes[1, 0].grid(True, axis="y", alpha=0.3)
label_offset = max(equipment_counts) * 0.01 if max(equipment_counts) > 0 else 0.1
for bar, count in zip(bars, equipment_counts):
    axes[1, 0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + label_offset,
        str(count),
        ha="center",
        va="bottom",
    )

axes[1, 1].remove()
profile_ax = fig.add_subplot(2, 2, 4, projection="polar")
profile_names = ["Cost efficiency", "Productivity", "Flexibility", "Low energy use"]
profile_values = [
    min(1.0, 1.0 / (1.0 + total_cost)),
    min(1.0, total_productivity / 1000.0),
    min(1.0, weighted_flexibility),
    min(1.0, 1.0 / (1.0 + average_energy)),
]
angles = np.linspace(0, 2 * np.pi, len(profile_names), endpoint=False)
closed_angles = np.append(angles, angles[0])
closed_values = profile_values + [profile_values[0]]
profile_ax.plot(closed_angles, closed_values, marker="o", color="purple", linewidth=2)
profile_ax.fill(closed_angles, closed_values, color="purple", alpha=0.2)
profile_ax.set_xticks(angles)
profile_ax.set_xticklabels(profile_names)
profile_ax.set_ylim(0, 1)
profile_ax.set_title("Scaled Outcome Profile", pad=20)

plt.tight_layout()
plt.show()

### Solution feasibility and objective breakdown

A useful interpretation separates the observed equipment mix from the score that guided the search. The table below reports the physical quantities, each weighted objective contribution, the total penalty, and the final fitness. This makes it possible to see whether a high score is associated with a feasible solution and which modeled trade-offs contribute to the score.

In [ ]:
def evaluate_solution_comprehensive(
    solution: Dict[str, int],
    equipment_types: Dict[str, EquipmentType],
    optimizer: ArtificialBeeColonyOptimizer,
) -> Dict[str, float]:
    """Calculate model quantities and fitness components for integer counts."""
    if set(solution) != set(equipment_types):
        raise ValueError("solution must contain one count for each equipment type")
    if any(not isinstance(count, (int, np.integer)) or count < 0 for count in solution.values()):
        raise ValueError("equipment counts must be nonnegative integers")

    total_units = sum(solution.values())
    if total_units == 0:
        return {
            "cost": 0.0,
            "productivity": 0.0,
            "flexibility": 0.0,
            "average_energy": 0.0,
            "sustainability": 0.0,
            "cost_contribution": 0.0,
            "productivity_contribution": 0.0,
            "flexibility_contribution": 0.0,
            "sustainability_contribution": 0.0,
            "penalty": 1e6,
            "fitness": -1e6,
        }

    total_cost_value = sum(
        equipment_types[name].capital_cost * count for name, count in solution.items()
    )
    total_productivity_value = sum(
        equipment_types[name].productivity * count for name, count in solution.items()
    )
    flexibility_value = sum(
        equipment_types[name].flexibility * count for name, count in solution.items()
    ) / total_units
    average_energy_value = sum(
        equipment_types[name].energy_consumption * count
        for name, count in solution.items()
    ) / total_units
    sustainability_value = 1.0 / (1.0 + average_energy_value)

    contributions = {
        "cost_contribution": optimizer.objective_weights["cost"] / (1.0 + total_cost_value),
        "productivity_contribution": (
            optimizer.objective_weights["productivity"] * total_productivity_value / 1000.0
        ),
        "flexibility_contribution": (
            optimizer.objective_weights["flexibility"] * flexibility_value
        ),
        "sustainability_contribution": (
            optimizer.objective_weights["sustainability"] * sustainability_value
        ),
    }
    penalty = 0.0
    if total_cost_value > optimizer.budget_limit:
        penalty += (total_cost_value - optimizer.budget_limit) * 10
    if total_productivity_value < optimizer.min_productivity:
        penalty += (optimizer.min_productivity - total_productivity_value) * 5
    if total_units < 5:
        penalty += (5 - total_units) * 2

    fitness_value = sum(contributions.values()) - penalty
    return {
        "cost": total_cost_value,
        "productivity": total_productivity_value,
        "flexibility": flexibility_value,
        "average_energy": average_energy_value,
        "sustainability": sustainability_value,
        **contributions,
        "penalty": penalty,
        "fitness": fitness_value,
    }


abc_solution = {"RTG": best_rtg, "RMG": best_rmg, "Straddle": best_straddle}
abc_metrics = evaluate_solution_comprehensive(abc_solution, equipment_types, optimizer)
assert np.isclose(abc_metrics["fitness"], optimizer.best_fitness, rtol=1e-12, atol=1e-12)

constraint_status = {
    "Budget within limit": abc_metrics["cost"] <= optimizer.budget_limit,
    "Productivity requirement met": abc_metrics["productivity"] >= optimizer.min_productivity,
    "Minimum fleet size met": sum(abc_solution.values()) >= 5,
}
objective_table = pd.DataFrame(
    {
        "Measure": [
            "Capital cost (million USD)",
            "Productivity (containers/hour)",
            "Weighted average flexibility",
            "Average energy (kWh/container)",
            "Cost contribution",
            "Productivity contribution",
            "Flexibility contribution",
            "Sustainability contribution",
            "Constraint penalty",
            "Fitness",
        ],
        "Value": [
            abc_metrics["cost"],
            abc_metrics["productivity"],
            abc_metrics["flexibility"],
            abc_metrics["average_energy"],
            abc_metrics["cost_contribution"],
            abc_metrics["productivity_contribution"],
            abc_metrics["flexibility_contribution"],
            abc_metrics["sustainability_contribution"],
            abc_metrics["penalty"],
            abc_metrics["fitness"],
        ],
    }
)
print("=== MODEL SCORE BREAKDOWN ===")
print(objective_table.round(4).to_string(index=False))
print("\n=== FEASIBILITY CHECK ===")
for condition, is_satisfied in constraint_status.items():
    print(f"{condition}: {is_satisfied}")

### Colony-size sensitivity analysis

This experiment retains the ABC method and changes only the colony-size setting across the listed runs. Each setting uses the same iteration limit, abandonment limit, and convergence-window configuration. The random-number stream continues across runs, so this is a single sequence of illustrative runs rather than a replicated statistical experiment. Runtime and solution quality can vary with both the parameter setting and stochastic search history.

In [ ]:
colony_sizes = [20, 30, 40, 50, 60]
results_colony_size = []

print("=== COLONY-SIZE SENSITIVITY ANALYSIS ===\n")
for colony_size in colony_sizes:
    config_test = ABCConfig(
        colony_size=colony_size,
        max_iterations=200,
        abandonment_limit=40,
        convergence_window=15,
    )
    optimizer_test = ArtificialBeeColonyOptimizer(
        equipment_types=equipment_types,
        config=config_test,
        budget_limit=50.0,
        min_productivity=500,
    )

    start_time = time.perf_counter()
    best_solution_test, fitness_history_test, _ = optimizer_test.optimize()
    runtime = time.perf_counter() - start_time
    test_counts = [
        int(np.clip(value, lower, upper))
        for value, (lower, upper) in zip(
            best_solution_test,
            [optimizer_test.bounds["RTG"], optimizer_test.bounds["RMG"], optimizer_test.bounds["Straddle"]],
        )
    ]
    test_solution = dict(zip(["RTG", "RMG", "Straddle"], test_counts))
    test_metrics = evaluate_solution_comprehensive(
        test_solution,
        equipment_types,
        optimizer_test,
    )

    results_colony_size.append(
        {
            "Colony_Size": colony_size,
            "Best_Fitness": optimizer_test.best_fitness,
            "Iterations": len(fitness_history_test),
            "Runtime_sec": runtime,
            "Productivity": test_metrics["productivity"],
            "Total_Cost": test_metrics["cost"],
            "Converged": optimizer_test.convergence_iteration is not None,
        }
    )
    print(
        f"Colony size {colony_size}: fitness={optimizer_test.best_fitness:.4f}, "
        f"productivity={test_metrics['productivity']:.0f} containers/hour, "
        f"runtime={runtime:.2f} seconds"
    )

colony_df = pd.DataFrame(results_colony_size)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("ABC Colony-Size Sensitivity Analysis", fontsize=16, fontweight="bold")

axes[0, 0].plot(
    colony_df["Colony_Size"],
    colony_df["Best_Fitness"],
    marker="o",
    color="steelblue",
    linewidth=2,
)
axes[0, 0].set_xlabel("Colony size (bees)")
axes[0, 0].set_ylabel("Best fitness")
axes[0, 0].set_title("Observed Fitness by Colony Size")
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(
    colony_df["Colony_Size"],
    colony_df["Runtime_sec"],
    marker="o",
    color="darkorange",
    linewidth=2,
)
axes[0, 1].set_xlabel("Colony size (bees)")
axes[0, 1].set_ylabel("Runtime (seconds)")
axes[0, 1].set_title("Runtime by Colony Size")
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(
    colony_df["Colony_Size"],
    colony_df["Productivity"],
    marker="o",
    color="seagreen",
    linewidth=2,
)
axes[1, 0].set_xlabel("Colony size (bees)")
axes[1, 0].set_ylabel("Productivity (containers/hour)")
axes[1, 0].set_title("Productivity of Best Observed Mix")
axes[1, 0].grid(True, alpha=0.3)

converged_count = int(colony_df["Converged"].sum())
not_converged_count = len(colony_df) - converged_count
axes[1, 1].bar(
    ["Criterion met", "Criterion not met"],
    [converged_count, not_converged_count],
    color=["seagreen", "indianred"],
    alpha=0.8,
)
axes[1, 1].set_ylabel("Number of configurations")
axes[1, 1].set_title("Convergence-Criterion Outcomes")
axes[1, 1].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()
print("\nSensitivity results are available in colony_df.")

### Results summary and interpretation

The final cell summarizes the main run using values computed by the notebook. Interpret the convergence criterion narrowly: it indicates that the range of recent best-fitness values fell below the configured tolerance after the minimum iteration condition was met. It does not establish that the global optimum has been found. The sensitivity table likewise reports observed outcomes for these particular stochastic runs; repeated runs would be needed to characterize variability.

In [ ]:
print("=== ARTIFICIAL BEE COLONY RUN SUMMARY ===\n")
print("Problem: Container-terminal equipment selection")
print("Search method: Artificial Bee Colony optimization\n")

print("MAIN-RUN CONFIGURATION:")
print(f"  Colony size: {config.colony_size} bees")
print(f"  Maximum iterations: {config.max_iterations}")
print(f"  Abandonment limit: {config.abandonment_limit}")
print(f"  Convergence window: {config.convergence_window}\n")

print("BEST OBSERVED EQUIPMENT MIX:")
print(f"  RTG units: {best_rtg}")
print(f"  RMG units: {best_rmg}")
print(f"  Straddle units: {best_straddle}")
print(f"  Total equipment: {total_units} units\n")

print("MODEL OUTCOMES:")
print(f"  Capital cost: ${total_cost:.1f}M (budget: ${optimizer.budget_limit:.1f}M)")
print(
    f"  Productivity: {total_productivity:.0f} containers/hour "
    f"(requirement: {optimizer.min_productivity:.0f})"
)
print(f"  Weighted average flexibility: {weighted_flexibility:.2f}")
print(f"  Average energy consumption: {average_energy:.2f} kWh/container")
print(f"  Fitness: {optimizer.best_fitness:.4f}\n")

print("SEARCH DIAGNOSTICS:")
print(f"  Iterations recorded: {len(fitness_history)}")
if optimizer.convergence_iteration is None:
    print("  Convergence criterion: not met during this run")
else:
    print(f"  Convergence criterion first met at iteration: {optimizer.convergence_iteration}")
print(f"  Final population diversity: {diversity_history[-1]:.3f}")
print(f"  Best-fitness change across recorded iterations: {fitness_history[-1] - fitness_history[0]:.4f}\n")

print("INTERPRETATION:")
print("  Check the feasibility report before treating the mix as usable.")
print("  The weighted score reflects the stated weights and penalty magnitudes.")
print("  ABC returns the best candidate observed in a stochastic search, not an optimality certificate.")
print("  The colony-size experiment is illustrative; its outcomes are specific to these runs.")
print("  Additional operational constraints can be represented by extending the model assumptions and score.")